# # Progress tracking, visually

Everything incremental in this library turns on a small vocabulary — **lattice**, **antichain**, **frontier** — together with the consumer-side boundary that `Evaluator.gc()` uses to reclaim memory.

This notebook introduces each term on a 6×6 `Time2D` grid and then runs a real Datalog circuit so the abstractions land against a non-trivial DAG.


In [ ]:
import types

import pandas as pd
from plotnine import (
    ggplot, aes, geom_tile, geom_point, geom_text, geom_col, geom_step,
    scale_fill_manual, scale_color_manual, scale_fill_gradient, scale_x_continuous, scale_y_continuous,
    theme_bw, theme, element_blank, labs, facet_wrap, coord_equal,
    position_dodge,
)

from pydbsp.algorithms.datalog import IncrementalDatalog, saturate
from pydbsp.algorithms import Variable
from pydbsp.core import Antichain, Time2D
from pydbsp.evaluator import Evaluator
from pydbsp.zset import ZSet


def grid_df(n_outer: int = 6, n_inner: int = 6) -> pd.DataFrame:
    return pd.DataFrame(
        [(o, k) for o in range(n_outer) for k in range(n_inner)],
        columns=["outer", "inner"],
    )


def base_tile(df):
    return (
        ggplot(df, aes("outer", "inner"))
        + coord_equal()
        + scale_x_continuous(breaks=sorted(df.outer.unique()))
        + scale_y_continuous(breaks=sorted(df.inner.unique()))
        + theme_bw()
        + theme(panel_grid_minor=element_blank())
    )

## 1. Lattice

The time type here is `Time2D` — `DBSPTime(nestedness=2)` — the componentwise-product order on ℕ × ℕ. Each cell on the grid is one admissible timestamp `(outer, inner)`. The partial order is: `(a, b) ≤ (c, d)` iff `a ≤ c` **and** `b ≤ d` — strictly weaker than a total order (e.g. `(1, 3)` and `(3, 1)` are incomparable).

In [ ]:
df = grid_df(6, 6)
(base_tile(df) + geom_tile(fill="white", color="grey") + labs(title="Time2D lattice"))

## 2. Antichain

An **antichain** is a set of pairwise-incomparable points. Its **downset** (the set of points dominated by some antichain element) is the settled region of a stream at a given moment; the antichain itself is the **frontier**.

Here's an antichain `{(1, 3), (2, 2), (3, 1)}` (red dots) and the downset it bounds (shaded).

In [ ]:
frontier = [(1, 3), (2, 2), (3, 1)]
front_df = pd.DataFrame(frontier, columns=["outer", "inner"])
A: Antichain = Antichain(Time2D)
for pt in frontier:
    A.insert(pt)

df = grid_df(6, 6)
df["settled"] = [A.covers((o, k)) for o, k in zip(df.outer, df.inner)]

(
    base_tile(df)
    + geom_tile(aes(fill="settled"), color="grey")
    + geom_point(data=front_df, mapping=aes("outer", "inner"), color="red", size=4)
    + scale_fill_manual(values={True: "#cce5ff", False: "white"})
    + labs(title="Antichain (red) + downset (blue) = settled region")
)

## 3. A real circuit — Datalog transitive closure

Reach's DAG is small (~15 ops) and its fixpoint is 1-dimensional (only the inner axis iterates). To see the scheduling story clearly we need something bigger — the positive Datalog interpreter running a single TC rule over a 3-edge graph has ~80 operators and 28 topological layers.

We build it, saturate the fixpoint, then collect the full transitive-deps closure of `observable.at((0,))` — the **work set** the evaluator materialises for one external read.

In [ ]:
program = ZSet(
    {
        (
            ("T", (Variable("X"), Variable("Y"))),
            ("E", (Variable("X"), Variable("Y"))),
        ): 1,
        (
            ("T", (Variable("X"), Variable("Z"))),
            ("E", (Variable("X"), Variable("Y"))),
            ("T", (Variable("Y"), Variable("Z"))),
        ): 1,
    }
)
edb = ZSet({("E", (0, 1)): 1, ("E", (1, 2)): 1, ("E", (2, 3)): 1})

c = IncrementalDatalog()
c.edb.push((0,), edb)
c.program.push((0,), program)
c.evaluator.gc = lambda: 0  # disable auto-gc so the full work set survives
saturate(c)
_ = c.observable_at((0,))  # materialise everything the root needs

ev = c.evaluator
op_idx = ev._op_idx
schedule = ev._schedule
op_to_layer: dict[int, int] = {}
for l_idx, layer in enumerate(schedule.layers):
    for op_i in layer:
        op_to_layer[op_i] = l_idx


def collect_work(ev, root_op, root_t):
    """Transitive deps of ``root_op.at(root_t)`` — the work set + dep edges."""
    work: set[tuple[int, tuple]] = set()
    deps_map: dict[tuple[int, tuple], list[tuple[int, tuple]]] = {}
    stack = [(root_op, root_t)]
    while stack:
        op, t = stack.pop()
        key = (op_idx[id(op)], t)
        if key in work:
            continue
        work.add(key)
        ds = []
        for dep_op, dep_t in op.deps(t):
            dkey = (op_idx[id(dep_op)], dep_t)
            ds.append(dkey)
            stack.append((dep_op, dep_t))
        deps_map[key] = ds
    return work, deps_map


work, deps_map = collect_work(ev, c.observable, (0,))
print(f"ops in schedule: {len(schedule.ops)}  layers: {len(schedule.layers)}")
print(f"work set for observable.at((0,)): {len(work)} cells, {sum(len(v) for v in deps_map.values())} dep edges")

## 4. The dual of the frontier — what GC can reclaim

`Evaluator.gc()` walks the consumer-intersection antichain: for each operator `X`, intersect across downstream consumers `C` the set of `X`-cells that `C`'s already-computed `t_C` values depend on. That intersection is the set of cells guaranteed to never be re-requested — safe to evict.

In [ ]:
raw = {k for k in ev.slots.keys()}
ev.gc = types.MethodType(Evaluator.gc, ev)  # restore the real gc
ev.gc()
survived = set(ev.slots.keys())
evicted = raw - survived

dual_df = pd.DataFrame(
    [
        {
            "op_idx": op_idx.get(oid, -1),
            "d": sum(t),
            "state": "evicted" if (oid, t) in evicted else "kept",
        }
        for (oid, t) in raw
    ]
)

print(f"raw slots: {len(raw)}  kept: {len(survived)}  evicted: {len(evicted)}")

(
    ggplot(dual_df, aes("d", "op_idx"))
    + geom_tile(aes(fill="state"), color="grey", size=0.2)
    + scale_fill_manual(values={"kept": "#a8d8a8", "evicted": "#dd6666"})
    + theme_bw()
    + theme(panel_grid_minor=element_blank(), figure_size=(9, 6))
    + labs(
        title="After gc() — red cells were antichain-algebraically safe to drop",
        x="sum(t)",
        y="operator index",
        fill="per-cell status",
    )
)

## Summary

- **Lattice** — time type; here `ℕ × ℕ` under componentwise order.
- **Antichain** — a set of pairwise-incomparable points; the lattice's notion of a maximal boundary.
- **Frontier** — the antichain of timestamps a stream considers settled; grows monotonically.
- **P-antichain dispatch** — what the `Evaluator` ships: at each step, dispatch every cell whose deps are already computed.
- **Consumer-intersection boundary** — per-operator cells every consumer has already passed; what `gc()` reclaims under the monotone-forward-deps contract.

All of this is one data structure — the `(op_id, t) → value` slot table — observed through different lenses.
